# CCTV Safety — Stage 2 Temporal Event Classification (Fight Detection)
## Google Colab Preparation, Packaging, & Verification Specification

> **Pipeline Version:** Two-Stage Safety Pipeline v2.0  
> **Stage 1 Spatial Model:** YOLOv8 6-class object detector (Person, Helmet, Vest, Fall, Fire, Smoke)  
> **Stage 2 Target:** Temporal Video Action Classifier (Binary: `0: non_fight`, `1: fight`)  
> **Dataset Package:** `scfd_pilot` (300 clips: 210 train, 45 val, 45 test; all 300 clips `OWNER_HUMAN_QA_APPROVED`)  
> **Mandatory Governance Guard:** `TRAINING_APPROVED = False`. Execution strictly halts before any optimizer creation, model construction, or training loop.

## 1. Local-to-Google Drive Packaging & Upload Instructions

The SCFD pilot dataset package was materialized locally under `data/processed/scfd_pilot/` (87.30 MB across 300 clips and manifests).  
**Before running this notebook in Colab, the package must be uploaded to your Google Drive.**

### Step 1: Create a Zip Archive Locally (Terminal / PowerShell)
From your local repository root (`Project-CCTV-Safety-antigravity`), run:
```powershell
Compress-Archive -Path data/processed/scfd_pilot -DestinationPath scfd_pilot_package.zip
```

### Step 2: Upload to Google Drive
1. Open [Google Drive](https://drive.google.com).
2. Navigate to or create your project directory (e.g. `MyDrive/Project-CCTV-Safety/`).
3. Upload `scfd_pilot_package.zip` and extract it so that its contents reside at:  
   `MyDrive/Project-CCTV-Safety/data/processed/scfd_pilot/`
4. Confirm the directory contains:
   - `scfd_pilot_manifest.json`
   - `train_manifest.csv`, `val_manifest.csv`, `test_manifest.csv`
   - Subdirectories: `train/` (210 clips), `val/` (45 clips), `test/` (45 clips).

## 2. Google Drive Mounting & Environment Setup
Mount Google Drive and configure `DRIVE_DATASET_ROOT`. When executed in local development, it automatically falls back to local repository paths.

In [ ]:
import os
import sys
import json
from pathlib import Path

# Detect execution environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Set your Google Drive dataset path:
    DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/Project-CCTV-Safety/data/processed/scfd_pilot")
else:
    # Local fallback for inspection and static validation
    DRIVE_DATASET_ROOT = Path("data/processed/scfd_pilot")

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Dataset root: {DRIVE_DATASET_ROOT.resolve()}")

## 3. Package Verification & Leakage-Free Dataset Gate
Verifies:
1. Canonical JSON manifest (`scfd_pilot_manifest.json`) and split CSVs exist and are readable.
2. All 300 materialized clips (`210 train`, `45 val`, `45 test`) exist on disk.
3. Binary class distribution across splits (verifying 1:1 balance in train, and ~1:1 in val/test).
4. Zero cross-split scene group leakage across all 253 connected components.
5. Atomicity of the 7 Owner-Signed cross-label clusters (Clusters A–G).

In [ ]:
import csv
from collections import Counter, defaultdict

assert DRIVE_DATASET_ROOT.is_dir(), f"Dataset directory not found: {DRIVE_DATASET_ROOT}"

manifest_json_path = DRIVE_DATASET_ROOT / "scfd_pilot_manifest.json"
assert manifest_json_path.is_file(), f"Missing canonical manifest: {manifest_json_path}"

with open(manifest_json_path, "r", encoding="utf-8") as f:
    manifest_meta = json.load(f)

print(f"Loaded manifest: {manifest_meta['dataset_name']} v{manifest_meta['dataset_version']}")
print(f"Total documented clips: {manifest_meta['total_clips']}")

# Load and verify split manifests
split_clips = {}
split_counts = {s: Counter() for s in ("train", "val", "test")}
clip_to_group = {}
clip_to_split = {}

for s in ("train", "val", "test"):
    csv_path = DRIVE_DATASET_ROOT / f"{s}_manifest.csv"
    assert csv_path.is_file(), f"Missing split CSV: {csv_path}"
    
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        split_clips[s] = rows
        for row in rows:
            cid = row["clip_id"]
            lbl = row["binary_source_label"]
            gid = row["scene_group_id"]
            split_counts[s][lbl] += 1
            clip_to_group[cid] = gid
            clip_to_split[cid] = s
            
            # Verify materialized video clip file exists
            clip_file = DRIVE_DATASET_ROOT / s / f"{cid}.mp4"
            assert clip_file.is_file(), f"Missing materialized clip file: {clip_file}"

print("\n--- Verified Per-Split Counts & Binary Balance ---")
for s in ("train", "val", "test"):
    f_cnt = split_counts[s]["fight"]
    nf_cnt = split_counts[s]["non_fight"]
    tot = f_cnt + nf_cnt
    print(f"  {s:>5}: {tot:3d} clips | Fight: {f_cnt:3d} | Non-Fight: {nf_cnt:3d} | Target: {manifest_meta['splits'][s]}")

# Verify zero cross-split scene group leakage
groups_by_split = defaultdict(set)
for cid, s in clip_to_split.items():
    groups_by_split[s].add(clip_to_group[cid])

assert not (groups_by_split["train"] & groups_by_split["val"]), "LEAKAGE: Train and Val share scene groups!"
assert not (groups_by_split["train"] & groups_by_split["test"]), "LEAKAGE: Train and Test share scene groups!"
assert not (groups_by_split["val"] & groups_by_split["test"]), "LEAKAGE: Val and Test share scene groups!"
print("\nPASS: Zero cross-split scene group leakage across all 253 scene components.")

# Verify Owner-Approved Cross-Label Clusters (Clusters A-G)
OWNER_CLUSTERS = {
    "Cluster_A": ["scfd_fight_fi046", "scfd_fight_fi047", "scfd_fight_fi049", "scfd_fight_fi050", "scfd_non_fight_nofi023", "scfd_non_fight_nofi024"],
    "Cluster_B": ["scfd_fight_fi065", "scfd_fight_fi066", "scfd_fight_fi067", "scfd_non_fight_nofi097", "scfd_non_fight_nofi098"],
    "Cluster_C": ["scfd_fight_fi073", "scfd_fight_fi074", "scfd_non_fight_nofi147"],
    "Cluster_D": ["scfd_fight_fi059", "scfd_non_fight_nofi148"],
    "Cluster_E": ["scfd_fight_fi107", "scfd_non_fight_nofi027"],
    "Cluster_F": ["scfd_fight_fi116", "scfd_non_fight_nofi051"],
    "Cluster_G": ["scfd_fight_fi122", "scfd_fight_fi123", "scfd_non_fight_nofi055", "scfd_non_fight_nofi056"],
}
for cname, members in OWNER_CLUSTERS.items():
    assigned_splits = {clip_to_split[m] for m in members}
    assert len(assigned_splits) == 1, f"Owner cluster {cname} split across {assigned_splits}!"
    print(f"PASS: {cname} ({len(members)} clips) atomically confined to split '{list(assigned_splits)[0]}'")

## 4. Explicit Configuration & Unresolved Architectural Choices

> [!WARNING]
> **OWNER_INPUT_NEEDED:** The following two core architecture choices remain unresolved.
> In accordance with project policy, they are **not** silently chosen or hardcoded to defaults.

1. **`BACKBONE_ARCHITECTURE` (`OWNER_INPUT_NEEDED`):**  
   The temporal video classification backbone. Candidate options under evaluation:
   - `x3d_s`: Preferred for edge surveillance and real-time inference (lightweight, efficient).
   - `r3d_18`: Standard 18-layer spatiotemporal ResNet baseline.
   - `mc3_18`: Mixed 3D convolution (2D spatial + 1D temporal).
   - `videomae`: Pretrained video transformer (high capacity, requires more GPU memory).

2. **`INPUT_MODALITY` (`OWNER_INPUT_NEEDED`):**  
   The spatial framing fed into the temporal classifier:
   - `full_frame`: Standard surveillance video frame ($224 \times 224$ uniform temporal sampling).
   - `stage1_person_crop`: Bounding-box crops around interacting person detections output by Stage 1 YOLOv8 Person detector + ByteTrack.

In [ ]:
# ==============================================================================
# CONFIGURATION CELL — UNRESOLVED CHOICES (OWNER INPUT NEEDED)
# ==============================================================================

# 1. Temporal Video Backbone Architecture
# Pending Owner Decision: Select one of ['x3d_s', 'r3d_18', 'mc3_18', 'videomae']
BACKBONE_ARCHITECTURE = None  # OWNER_INPUT_NEEDED

# 2. Input Modality / Framing
# Pending Owner Decision: Select one of ['full_frame', 'stage1_person_crop']
INPUT_MODALITY = None  # OWNER_INPUT_NEEDED

# Preprocessing specification (agreed pilot baseline)
FRAMES_PER_CLIP = 16
FRAME_RESOLUTION = (224, 224)
CLIP_DURATION_SEC = 2.0
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("=== Stage 2 Pilot Configuration Status ===")
print(f"  BACKBONE_ARCHITECTURE: {BACKBONE_ARCHITECTURE} (OWNER_INPUT_NEEDED)")
print(f"  INPUT_MODALITY:        {INPUT_MODALITY} (OWNER_INPUT_NEEDED)")
print(f"  FRAMES_PER_CLIP:       {FRAMES_PER_CLIP}")
print(f"  FRAME_RESOLUTION:      {FRAME_RESOLUTION}")
print(f"  CLIP_DURATION_SEC:     {CLIP_DURATION_SEC}")

if BACKBONE_ARCHITECTURE is None or INPUT_MODALITY is None:
    print("\n[ACTION REQUIRED] BACKBONE_ARCHITECTURE and INPUT_MODALITY require explicit Project Owner decision before training.")

## 5. Mandatory Governance Hard Guard (`TRAINING_APPROVED = False`)

> [!CAUTION]
> **HARD EXECUTION GUARD:**  
> `TRAINING_APPROVED` is strictly `False`.  
> Per project governance, Stage 2 model training remains **HOLD / BLOCKED** pending:
> 1. Stage 1 YOLOv8 spatial detector baseline formal verification in Colab.
> 2. Project Owner resolution of `BACKBONE_ARCHITECTURE` and `INPUT_MODALITY`.
> 3. Explicit written authorization from the Project Owner to initiate Stage 2 training.
>
> When **Run all** is executed in Colab, the code cell below **halts execution immediately** before any optimizer creation, model construction, or training loop.
> No optimizer is instantiated, no model weights are trained, and no GPU training compute is consumed.

In [ ]:
# ==============================================================================
# MANDATORY GOVERNANCE HARD GUARD
# ==============================================================================
TRAINING_APPROVED = False

if not TRAINING_APPROVED:
    print("\n" + "!" * 80)
    print("HALT: TRAINING_APPROVED is False.")
    print("Stage 2 model training, optimizer instantiation, and inference are strictly BLOCKED.")
    print("Execution halted safely before optimizer creation or training.")
    print("!" * 80 + "\n")
    raise SystemExit("STOP: Stage 2 training is blocked under project governance (TRAINING_APPROVED == False).")

## 6. Colab Quota-Safe Checkpoint-to-Drive & Interruption Guidance

*(All sections below are disabled templates that will only execute when `TRAINING_APPROVED` is set to `True` following Owner Authorization).*

### Colab Session & Quota Best Practices:
1. **Drive Checkpointing:** Save model checkpoints directly to Google Drive (`DRIVE_DATASET_ROOT / "checkpoints" / "checkpoint_epoch_XX.pt"`). If a Colab session disconnects or reaches GPU time limits, trained weights remain persisted in Drive.
2. **Explicit Resume Template:** Never auto-resume into an untracked training loop. A resume cell must explicitly inspect the saved checkpoint, verify epoch counter, loss, and optimizer state before resuming.
3. **Execution Order (When Authorized):**
   - Step 1: Resolve `BACKBONE_ARCHITECTURE` and `INPUT_MODALITY` in Section 4.
   - Step 2: Set `TRAINING_APPROVED = True` in Section 5 with Project Owner authorization.
   - Step 3: Run Section 7 to initialize Dataset, Model, and Optimizer.
   - Step 4: Execute Section 8 training loop with checkpointing.
   - Step 5: Run Section 9 for held-out test evaluation.

## 7. Model Construction & Optimizer Setup (Disabled Template)
This cell is strictly gated behind `TRAINING_APPROVED`. When `TRAINING_APPROVED` is `False`, it cannot be reached.

In [ ]:
# ==============================================================================
# DISABLED TEMPLATE: MODEL & OPTIMIZER CONSTRUCTION (GATED)
# ==============================================================================
if not TRAINING_APPROVED:
    raise RuntimeError("Access denied: TRAINING_APPROVED is False.")

if BACKBONE_ARCHITECTURE is None or INPUT_MODALITY is None:
    raise ValueError("Cannot construct model: BACKBONE_ARCHITECTURE and INPUT_MODALITY must be decided by Project Owner.")

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models.video import r3d_18, R3D_18_Weights

# Example instantiation template for approved backbone
print(f"Building approved Stage 2 model: {BACKBONE_ARCHITECTURE} ({INPUT_MODALITY})...")
if BACKBONE_ARCHITECTURE == "r3d_18":
    model = r3d_18(weights=R3D_18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_features, 2))
else:
    raise NotImplementedError(f"Backbone '{BACKBONE_ARCHITECTURE}' loader template not configured.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)

print(f"Model and optimizer successfully constructed on {device}.")

## 8. Training Execution & Drive Checkpointing Loop (Disabled Template)
This cell is strictly gated behind `TRAINING_APPROVED`.

In [ ]:
# ==============================================================================
# DISABLED TEMPLATE: TRAINING EXECUTION LOOP (GATED)
# ==============================================================================
if not TRAINING_APPROVED:
    raise RuntimeError("Access denied: TRAINING_APPROVED is False.")

CHECKPOINT_DIR = DRIVE_DATASET_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = CHECKPOINT_DIR / "best_stage2_model.pt"

print(f"Checkpoint destination configured: {CHECKPOINT_DIR}")
print("Training loop template available for execution once authorized.")

## 9. Held-Out Test Evaluation & Safety Metrics (Disabled Template)
This cell is strictly gated behind `TRAINING_APPROVED`.

In [ ]:
# ==============================================================================
# DISABLED TEMPLATE: TEST EVALUATION (GATED)
# ==============================================================================
if not TRAINING_APPROVED:
    raise RuntimeError("Access denied: TRAINING_APPROVED is False.")

print("Test evaluation template available once model training is completed.")

## 10. Governance Disclaimers & Production Deployment Notes
1. **Owner Human Visual QA:** All 300 clips carry genuine owner human visual QA approval (`OWNER_HUMAN_QA_APPROVED`). Raw rights/provenance remain unverified (`UNVERIFIED — OWNER-ACCEPTED EDUCATIONAL RISK`). Model training is strictly gated (`TRAINING_APPROVED = False`).
2. **Pilot Dataset Scope:** This package comprises 300 clips (87.30 MB). Validation metrics evaluate pilot capability and scene generalization, but do not replace target surveillance camera calibration.
3. **Hard-Negative Evaluation:** Before production release, the model must be benchmarked against false-positive hard negatives (e.g. fitness gyms, crowded elevators, bus boarding).
4. **Stage 1 Spatial Trigger Dependency:** Under the Two-Stage Safety Pipeline architecture, Stage 2 is invoked only after the Stage 1 YOLOv8 Person detector and ByteTrack register an anomaly candidate trigger.